# 07 · INT8 calibration pilot (6 engines)

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 1

**Purpose.** Build natural / no-print / no-replay INT8 engines for seeds 1 and 2 from the same frozen ONNX graph, audit their quantization scales, and evaluate both threshold policies.

**In plain words.** INT8 stores activations with 256 levels; calibration picks the range each layer uses by looking at 600 training images. We change only *which* images (the class mixture) and see whether the per-layer ranges and the attack-specific errors move. No weights are trained here.

**Expected inputs**
- Passed `equivalence.json`; ONNX model; processed train calibration frames; dev/test evaluation frames; FP32 `threshold.json`.

**Expected outputs**
- `QPAD_Project/results/int8/calibration_manifest.csv` (saved before any engine)
- `QPAD_Project/engines/int8_entropy_<condition>_s<seed>/` (engine, cache, scales, logs, metadata)
- `QPAD_Project/results/int8/<tag>/` (scores, thresholds, metrics)
- `pilot_summary.csv`

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
DRY_RUN_EVAL_VIDEOS_PER_CLASS = 3   # dry run: evaluate on 3 videos/class/split, 1 engine
CONTINUE_FULL_EXPERIMENT = False    # decision after inspecting this pilot; see the summary cell
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb07", paths.logs / "nb07.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

In [ ]:
# TensorRT is needed from here on. Its version is pinned in configs/int8_experiments.yaml: TensorRT 11
# removed the INT8 entropy calibrator that the pre-registered method uses. Colab preinstalls TensorRT 11,
# so the first time this cell swaps it for the pinned version and asks for ONE session restart.
TRT_PIN = str(cfgs["int8"]["tensorrt"]["version"])
def installed_trt():   # checked in a fresh process: an already-imported module would hide a reinstall
    r = subprocess.run([sys.executable, "-c", "import tensorrt; print(tensorrt.__version__)"],
                       capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else None
if IN_COLAB and installed_trt() != TRT_PIN:
    import torch
    cuda_major = torch.version.cuda.split(".")[0]
    old = [p["name"] for p in json.loads(subprocess.run([sys.executable, "-m", "pip", "list", "--format=json"],
                                                        capture_output=True, text=True, check=True).stdout)
           if p["name"].lower().replace("_", "-").startswith("tensorrt")]
    if old:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *old], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"tensorrt-cu{cuda_major}=={TRT_PIN}"], check=True)
    if installed_trt() != TRT_PIN:
        raise RuntimeError(f"installing TensorRT {TRT_PIN} failed (found {installed_trt()})")
    if "tensorrt" in sys.modules:
        raise RuntimeError(f"TensorRT {TRT_PIN} is now installed. Runtime -> Restart session, then Run all again "
                           "(needed once per runtime; the old version is still loaded in this session).")
try:
    import tensorrt as trt
except ImportError:
    trt = None
print("TensorRT:", getattr(trt, "__version__", None))
if trt is not None and trt.__version__ != TRT_PIN:
    raise RuntimeError(f"TensorRT {trt.__version__} is loaded but {TRT_PIN} is pinned: restart the session.")
from qpad.tensorrt_utils import hardware_fingerprint
HW = hardware_fingerprint(); print(HW)

In [ ]:
icfg = cfgs["int8"]
METHOD = icfg["calibration_method"]["primary"]   # pre-registered: entropy
SEEDS = icfg["seeds"]["pilot"][:1] if DRY_RUN else icfg["seeds"]["pilot"]
CONDS = ["natural"] if DRY_RUN else icfg["pilot_conditions"]
print("method:", METHOD, "| seeds:", SEEDS, "| conditions:", CONDS)
if not __import__("torch").cuda.is_available():
    raise RuntimeError("INT8 TensorRT engines need a CUDA GPU runtime.")
from qpad.experiments import build_calibration_manifest
CAL, CAL_STORE, CAL_PATH = build_calibration_manifest(paths, cfgs, DRY_RUN, SEEDS, CONDS)
print("calibration manifest saved BEFORE building engines:", CAL_PATH)
display(CAL.groupby(["seed", "condition", "class"]).size().unstack())

In [ ]:
import json, pandas as pd
from qpad.datasets import materialize
from qpad.experiments import (balanced_subset, build_int8_engine, calibration_array, engine_is_complete,
                              engine_tag, evaluate_backend)
from qpad.pipeline import processed_split_dir, find_input
from qpad.tensorrt_utils import TrtRunner
norm = cfgs["preprocessing"]["normalization"]
from qpad.data_exceptions import expected_eval_counts; nfr = expected_eval_counts(cfgs)
n_eval = DRY_RUN_EVAL_VIDEOS_PER_CLASS if DRY_RUN else None
DEV = balanced_subset(materialize(processed_split_dir(paths, DRY_RUN, "dev"), "dev", "eval", OUT.local_cache), n_eval)
TEST = balanced_subset(materialize(processed_split_dir(paths, DRY_RUN, "test"), "test", "eval", OUT.local_cache), n_eval)
FP32_THR = json.loads(find_input(paths, DRY_RUN, "results", "fp32/threshold.json").read_text())["threshold"]
preflight("INT8 pilot engines", len(SEEDS) * len(CONDS), [OUT.models / "deeppixbis_fp32.onnx", CAL_PATH],
          [OUT.engines, OUT.results / "int8"], LOCAL_WORK_ROOT, {"method": METHOD, **icfg["tensorrt"]}, SEEDS, DRY_RUN,
          "~2-5 min per engine build + scoring")

In [ ]:
PILOT = {}
for seed in SEEDS:
    for cond in CONDS:
        tag = engine_tag(METHOD, cond, seed)
        edir, rdir = OUT.engines / tag, OUT.results / "int8" / tag
        if engine_is_complete(edir, rdir) and not FORCE_REBUILD:
            print("verified, skipping", tag)
        else:
            imgs = calibration_array(CAL_STORE, CAL, seed, cond, norm)
            meta = build_int8_engine(paths, cfgs, DRY_RUN, imgs, tag, METHOD)
            evaluate_backend(TrtRunner(edir / "engine.plan"), DEV, TEST, norm, FP32_THR, rdir, nfr,
                             int(icfg["tensorrt"]["opt_batch"]))
        PILOT[tag] = json.loads((rdir / "metrics.json").read_text())
        print(tag, "policy B print", PILOT[tag]["policy_B"]["print_apcer"]["text"],
              "replay", PILOT[tag]["policy_B"]["replay_apcer"]["text"])

## Calibration-range audit
If scales are nearly identical across conditions, large performance differences are unlikely. Compare each engine with natural of the same seed.

In [ ]:
from qpad.calibration import compare_scales, scale_summary
scales = {}
for tag in PILOT:
    s = pd.read_csv(OUT.engines / tag / "scales.csv")
    scales[tag] = dict(zip(s.tensor, s.scale))
rows = []
for seed in SEEDS:
    ref = engine_tag(METHOD, "natural", seed)
    sub = {t: v for t, v in scales.items() if t.endswith(f"_s{seed}")}
    if ref in sub and len(sub) > 1:
        rows.append(scale_summary(compare_scales(sub, ref)).assign(reference=ref))
SCALES = pd.concat(rows) if rows else pd.DataFrame()
SCALES.to_csv(OUT.results / "int8" / "pilot_scale_audit.csv", index=False); display(SCALES)

In [ ]:
from qpad.metrics import metrics_rows
summary = []
for tag, mm in PILOT.items():
    for pol in ("A", "B"):
        summary.append({**metrics_rows(tag, mm[f"policy_{pol}"]), "policy": pol})
PILOT_SUMMARY = pd.DataFrame(summary)
PILOT_SUMMARY.to_csv(OUT.results / "int8" / "pilot_summary.csv", index=False)
display(PILOT_SUMMARY)
print("CONTINUE_FULL_EXPERIMENT =", CONTINUE_FULL_EXPERIMENT,
      "-> decide Tier 2A (full 35 engines) or Tier 2B (robustness) with your professor; do not start automatically.")

**Next notebook:** `08_full_int8_experiment.ipynb`